# ResNet-50 transfer learning on region crops (Colab, GPU)

Before running: in Google Drive, make a folder `indic_project` with `crops.zip` and `train_classifier.py` in it.

Runtime â†’ Change runtime type â†’ **T4 GPU**. Run the cells top to bottom. Results are written straight to Drive, so a disconnect does not lose finished runs.

In [ ]:
# 1. Check the GPU is on (should print a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Copy the crops to the Colab disk (reading 55k small files from Drive is very slow)
!unzip -q -o /content/drive/MyDrive/indic_project/crops.zip -d /content/
!ls /content/crops

# Work inside the Drive folder, so runs_cls/ is saved to Drive
%cd /content/drive/MyDrive/indic_project

## Step 5: transfer-learning strategies (C1 to C5)

Same settings for all five; only how much of ResNet-50 is unfrozen changes. Run one cell at a time. If the session disconnects, re-run cells 2 and 3, then continue with the next run.

In [ ]:
# C1: linear probe - only the new fc layer trains
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy linear --name C1_linear --data /content/crops --epochs 15 --patience 3 --workers 2

In [ ]:
# C2: partial - layer4 + fc train
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --name C2_layer4 --data /content/crops --epochs 15 --patience 3 --workers 2

In [ ]:
# C3: partial - layer3, layer4 + fc train
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer3 --name C3_layer3 --data /content/crops --epochs 15 --patience 3 --workers 2

In [ ]:
# C4: partial - layer2 to layer4 + fc train
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer2 --name C4_layer2 --data /content/crops --epochs 15 --patience 3 --workers 2

In [ ]:
# C5: full fine-tuning - everything trains
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy full --name C5_full --data /content/crops --epochs 15 --patience 3 --workers 2

## Step 6: overfitting study (on C2 = layer4 setup)

C2 overfits clearly (train 83% vs val 56%) and is fast, so every technique is tested on it, **one change at a time**. C2 itself is the baseline (no regularization), so it is not re-run.

Order = most important first, so if Colab runs out you still have the key ones. Compare each with C2 on: train-val gap, val loss curve, test macro-F1.

In [ ]:
# G2a: L2 (ridge) = weight_decay, small
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name G2a_l2_1e-4 --l2 1e-4

In [ ]:
# G2b: L2 (ridge), strong
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name G2b_l2_1e-2 --l2 1e-2

In [ ]:
# G5: dropout before the final layer
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name G5_dropout05 --dropout 0.5

In [ ]:
# G3: L1 (lasso); 1e-6 adds about 0.3 to a loss of about 1.5
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name G3_l1_1e-6 --l1 1e-6

In [ ]:
# G3b: L1 (lasso), stronger; 1e-5 adds about 3 to the loss
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name G3b_l1_1e-5 --l1 1e-5

In [ ]:
# L1 vs L2: what fraction of layer4 + fc weights ended up close to zero?
import os, torch
os.chdir('/content/drive/MyDrive/indic_project')
for run in ['C2_layer4', 'G2b_l2_1e-2', 'G3_l1_1e-6', 'G3b_l1_1e-5']:
    path = f'runs_cls/{run}/best_model.pth'
    if not os.path.exists(path):
        print(f'{run:14s} not run yet'); continue
    sd = torch.load(path, map_location='cpu')
    w = torch.cat([v.flatten() for k, v in sd.items() if (k.startswith('layer4') or k.startswith('fc')) and 'weight' in k and v.dim() > 1])
    print(f'{run:14s} mean |w| = {w.abs().mean():.5f}   |w| < 1e-3: {100*(w.abs() < 1e-3).float().mean():5.1f}%   |w| < 1e-4: {100*(w.abs() < 1e-4).float().mean():5.1f}%')

In [ ]:
# G10: data augmentation (rotation, brightness/contrast, blur; no flips)
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name G10_augment --augment

In [ ]:
# Imbalance check: class weights OFF (compare with C2, which has them ON)
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name C2_noweights --no-class-weights

In [ ]:
# G4: elastic net = L1 + L2 together
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name G4_elastic --l1 1e-6 --l2 1e-2

In [ ]:
# G7: remove max-pool after the first conv layer (about 4x more activations: slowest run, do it last)
!cd /content/drive/MyDrive/indic_project && python train_classifier.py --model resnet50 --strategy layer4 --data /content/crops --epochs 15 --patience 3 --workers 2 --name G7_nopool --no-first-pool

In [ ]:
# Results table so far
import pandas as pd
pd.read_csv('/content/drive/MyDrive/indic_project/runs_cls/summary.csv')[['name', 'strategy', 'trainable_params', 'best_epoch', 'train_acc', 'val_acc', 'gap', 'test_acc', 'test_macro_f1', 'sec_per_epoch']]